In [1]:
import pandas as pd
import numpy as np

In [2]:
data_path = '../data/raw/AmesHousing.csv'

df = pd.read_csv(data_path)
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [31]:
cols_wless_than_1pct = []

for col in df.columns:
    num_missing_values = df[col].isnull().sum()
    pct_missing_values = ( num_missing_values / len(df) ) * 100

    if pct_missing_values > 0:
        
        print(f'Col {col} - {pct_missing_values:.4f} % - {num_missing_values}/{len(df)}')
        
        if pct_missing_values < 1:
            cols_wless_than_1pct.append(col)

Col Lot Frontage - 16.7235 % - 490/2930
Col Alley - 93.2423 % - 2732/2930
Col Mas Vnr Type - 60.5802 % - 1775/2930
Col Mas Vnr Area - 0.7850 % - 23/2930
Col Bsmt Qual - 2.7304 % - 80/2930
Col Bsmt Cond - 2.7304 % - 80/2930
Col Bsmt Exposure - 2.8328 % - 83/2930
Col BsmtFin Type 1 - 2.7304 % - 80/2930
Col BsmtFin SF 1 - 0.0341 % - 1/2930
Col BsmtFin Type 2 - 2.7645 % - 81/2930
Col BsmtFin SF 2 - 0.0341 % - 1/2930
Col Bsmt Unf SF - 0.0341 % - 1/2930
Col Total Bsmt SF - 0.0341 % - 1/2930
Col Electrical - 0.0341 % - 1/2930
Col Bsmt Full Bath - 0.0683 % - 2/2930
Col Bsmt Half Bath - 0.0683 % - 2/2930
Col Fireplace Qu - 48.5324 % - 1422/2930
Col Garage Type - 5.3584 % - 157/2930
Col Garage Yr Blt - 5.4266 % - 159/2930
Col Garage Finish - 5.4266 % - 159/2930
Col Garage Cars - 0.0341 % - 1/2930
Col Garage Area - 0.0341 % - 1/2930
Col Garage Qual - 5.4266 % - 159/2930
Col Garage Cond - 5.4266 % - 159/2930
Col Pool QC - 99.5563 % - 2917/2930
Col Fence - 80.4778 % - 2358/2930
Col Misc Feature - 9

In [32]:
print(cols_wless_than_1pct)

['Mas Vnr Area', 'BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', 'Electrical', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Garage Cars', 'Garage Area']


In [42]:
df[df['BsmtFin SF 1'].isnull()]

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
1341,1342,903230120,20,RM,99.0,5940,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,4,2008,ConLD,Abnorml,79000


In [40]:
df[df['BsmtFin SF 2'].isnull()]

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
1341,1342,903230120,20,RM,99.0,5940,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,4,2008,ConLD,Abnorml,79000


In [19]:
def verify_structural_missingness(df, feature_name, quantitative_col, descriptive_cols):
    
    """
    Verifies if missing values in descriptive columns are structural 
    by checking if they align with a zero value in a quantitative column.
    """
    
    # Rows where the feature is not there (Count or Area == 0)
    absent_feature = df[df[quantitative_col] == 0] 
    
    # Check if all descriptive columns are correctly NaN for these rows
    all_null = absent_feature[descriptive_cols].isnull().all().all()
    
    # Identify mismatches (Garage Area is 0, but Garage Type is recorded)
    mismatches = absent_feature[absent_feature[descriptive_cols].notnull().any(axis=1)]
    
    print(f"--- Verification for {feature_name} ---")
    if all_null:
        print(f"SUCESS - All missing values appear to be Structural.")
    else:
        print(f"FAILED - Found {len(mismatches)} rows with potential data entry errors.")
    
    print(f"Total rows with 0 {feature_name}: {len(absent_feature)}\n")

In [ ]:
verification_groups = {
    "Pools": {
        "quantitative": "Pool Area", 
        "descriptive": ["Pool QC"]
    },
    "Fireplaces": {
        "quantitative": "Fireplaces", 
        "descriptive": ["Fireplace Qu"]
    },
    "Garages": {
        "quantitative": "Garage Area", 
        "descriptive": ["Garage Type", "Garage Finish", "Garage Qual", "Garage Cond"]
    },
    "Basements": {
        "quantitative": "Total Bsmt SF", 
        "descriptive": ["Bsmt Qual", "Bsmt Cond", "Bsmt Exposure", "BsmtFin Type 1", "BsmtFin Type 2"]
    },
    "Masonry Veneer": {
        "quantitative": "Mas Vnr Area", 
        "descriptive": ["Mas Vnr Type"]
    }
}

for feature, config in verification_groups.items():
    verify_structural_missingness(df, feature, config["quantitative"], config["descriptive"])

--- Verification for Pools ---
SUCESS - All missing values appear to be Structural.
Total rows with 0 Pools: 2917

--- Verification for Fireplaces ---
SUCESS - All missing values appear to be Structural.
Total rows with 0 Fireplaces: 1422

--- Verification for Garages ---
SUCESS - All missing values appear to be Structural.
Total rows with 0 Garages: 157

--- Verification for Basements ---
SUCESS - All missing values appear to be Structural.
Total rows with 0 Basements: 79

--- Verification for Masonry Veneer ---
FAILED - Found 3 rows with potential data entry errors.
Total rows with 0 Masonry Veneer: 1748



Most features that have missing values happen because the feature simply doesn't exist on the property, therefore most of the variables have **structural missing data**.

For example:
- Property doesn't have Pool
- Property doesn't have Fireplace
- Property doesn't have Garages
- Property doesn't have Basement